<a href="https://colab.research.google.com/github/mmferes/PPD-2026/blob/main/Aula3/aula_0704_Soma_linhas_matrizes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compartilhando dados entre as *threads*

Um dos aspectos para o uso de *threads*, ao invés de processos, na criação de aplicações que querem explorar o uso de múltiplos processadores no mesmo computador, é o compartilhamento de memória entre as *threads* de um processo.

De maneira simplificada, todas as **variáveis** que forem definidas como **globais**, ou seja, que forem **declaradas fora do escopo de qualquer função** no código, serão compartilhadas por todas as *threads* deste processo.

É claro, contudo, que se houver acessos simultâneos das *threads* sobre as mesmas posições de memória, pode ser preciso sincronizar esses acessos, evitando **condições de corrida** e erros. A garantida de acesso exclusivo, com exclusão mútua, por exemplo, não é provida automaticamente pelo SO. A aplicação deverá usar os mecanismos disponíveis para isso, se for o caso.


## Usando múltiplas *threads*

O programa a seguir ilustra o uso de *threads* para fazer a soma das linhas de 2 matrizes de forma paralela.

Para que sejam acessíveis por todas as *threads*, as matrizes foram declaradas globalmente.

Cabe uma discussão aqui, já que as matrizes serão alocadas dinamicamente.

Na verdade, as posições de memória contendo os endereços das matrizes, ou seja, os ponteiros, é que são globais. Neste exemplo, os espaços para as matrizes serão alocados dinamicamente. Vale saber, então, que as áreas de memória dinâmica (*heap*) também são acessíveis por todas as *threads* de um processo.

Neste exemplo específico, vale observar um aspecto relevante, que não é tratado no código, que é o cuidado com a divisão das linhas entre as *threads*. Para que o código seja flexível, com suporte a matrizes de tamanhos variados, ainda seria preciso tratar linhas restantes nos cálculos das divisões, para que não haja linhas sem manipulação!

In [ ]:
%%writefile t5.c
/*
 ** Programa :
 **   Uso de threads para soma das linhas de 2 matrizes
 */

#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/time.h>
#include <sys/resource.h>

#define LEN 128

#define NLIN 1000
#define NCOL 1000
#define NTHR 8

int *_A;
int *_B;
int *_C;

int _nlin = 0;
int _ncol = 0;
int _nthr = 0;

int _verbose = 0;

void * soma(void *arg)
{
  // determina o número lógico desta thread
	long int ind = (long int)arg;
	int i,j;
	int nl, li, lf;

	// Falta tratar divisão não inteira...
	nl = _nlin/_nthr;
	li = ind*nl;
	lf = li+nl;

  if (_verbose)
	  printf("Thread %ld manipula linhas %d a %d\n",ind,li,lf-1);

	for(i=li; i < lf; i++)

		for(j=0;j<_ncol;j++)

			_C[i*_ncol+j] = _A[i*_ncol+j] + _B[i*_ncol+j];

	pthread_exit(NULL);
}

int main (int argc, char *argv[])
{
	pthread_t threads[NLIN];
	long int t;
  int status;
	char err_msg[LEN];
	int i,j,ind;

	ind=1;
	while (ind < argc) {
		if( !strcmp(argv[ind],"-h") || !strcmp(argv[ind],"/?")) {
			printf("Uso: %s [-nc num_col] [-nl num_lin] [-nt num_thr] ",argv[0]);
			exit(0);
		}
    if(!strcmp(argv[ind],"-v"))
		  _verbose=1;

		if(!strcmp(argv[ind],"-nc")) {
			if(argc>ind)
				_ncol=atoi(argv[++ind]);
			else {
				printf("Erro nos parâmetros...\n");
				exit(0);
			}
		}
		if(!strcmp(argv[ind],"-nl")) {
			if(argc>ind)
				_nlin=atoi(argv[++ind]);
			else {
				printf("Erro nos parâmetros...\n");
				exit(0);
			}
		}
		if(!strcmp(argv[ind],"-nt")) {
			if(argc>ind)
				_nthr=atoi(argv[++ind]);
			else {
				printf("Erro nos parâmetros...\n");
				exit(0);
			}
		}

		ind++;
	}

	if(_nlin==0) _nlin = NLIN;
	if(_ncol==0) _ncol = NCOL;
	if(_nthr==0) _nthr = NTHR;

	// alocar as matrizes
	_A=(int *)malloc(_nlin * _ncol * sizeof(int));
	_B=(int *)malloc(_nlin * _ncol * sizeof(int));
	_C=(int *)malloc(_nlin * _ncol * sizeof(int));

  srand(getpid());

  // geração das matrizes com valores inteiros aleatórios (0 a 9). Senão, 0!
	for(i=0;i<_nlin;i++) {
		for(j=0;j<_ncol;j++) {
			_A[i*_ncol+j]=rand()%10;
			_B[i*_ncol+j]=rand()%10;
		}
	}

	if(_verbose) {
		printf("\n");
		for(i=0;i<_nlin;i++) {
			printf("%d: ",i);
			for(j=0;j<_ncol;j++)
				printf("%d ",_A[i*_ncol+j]);
			printf("   ");
			for(j=0;j<_ncol;j++)
				printf("%d ",_B[i*_ncol+j]);
			printf("\n");
		}
    printf("\n");
	}

  // cria threads, passando o índice lógico como parâmetro para cada uma delas
	for (t=0; t<_nthr; t++) {
		status = pthread_create(&threads[t], NULL, soma, (void *)t);
		if (status) {
			strerror_r(status,err_msg,LEN);
			printf("Falha da criacao da thread %ld (%d): %s\n",t,status,err_msg);
			exit(EXIT_FAILURE);
		}
	}

	// espera threads retornarem
	for (t=0; t<_nthr; t++) {

		status = pthread_join(threads[t], NULL);
		if (status) {
			strerror_r(status,err_msg,LEN);
			printf("Erro em pthread_join: %s\n",err_msg);
			exit(EXIT_FAILURE);
		}
	}

	// impressão da matriz resultante
	if(_verbose) {
		printf("\n");
		for(i=0;i<_nlin;i++) {
			printf("%d: ",i);
			for(j=0;j<_ncol;j++)
				printf("%2d ",_C[i*_nlin+j]);
			printf("\n");
		}
		printf("\n");
	}

	// desalocar as matrizes
	free(_A);
	free(_B);
	free(_C);

	return(0);
}

Overwriting t5.c


In [ ]:
! if [ ! t5 -nt t5.c ]; then gcc -Wall t5.c -o t5 -pthread ; fi
# ! time ./t5 -nl 10 -nc 10 -nt 5 -v
! lscpu | grep "CPU(s):"
! time ./t5 -nl 10240 -nc 10240 -nt 1
! time ./t5 -nl 10240 -nc 10240 -nt 2
! time ./t5 -nl 10240 -nc 10240 -nt 4
! time ./t5 -nl 10240 -nc 10240 -nt 8
! time ./t5 -nl 10240 -nc 10240 -nt 16
! time ./t5 -nl 10240 -nc 10240 -nt 32

CPU(s):                                  2
NUMA node0 CPU(s):                       0,1

real	0m6.345s
user	0m5.449s
sys	0m0.811s

real	0m5.459s
user	0m5.355s
sys	0m0.564s

real	0m6.023s
user	0m5.782s
sys	0m0.631s

real	0m5.447s
user	0m5.321s
sys	0m0.588s

real	0m6.092s
user	0m5.705s
sys	0m0.649s

real	0m5.485s
user	0m5.342s
sys	0m0.582s
